# DOE model outputs review and export notebook

Run this after `DOE_MASTER_FULL_PIPELINE.ipynb`. It does not retrain the model. It reads the local `outputs_runtime/` and `models_runtime/` folders created by the master notebook, then builds paper/slide-ready summary tables and figures.

This notebook is local-output focused: do not commit prediction rows, fitted model files, or approved-runtime exports back to GitHub unless reduced to public-safe summaries.

In [ ]:
from pathlib import Path
import json
import re
import warnings

import numpy as np
import pandas as pd
from joblib import load

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

OUTPUT_ROOT = Path.cwd() / 'outputs_runtime'
MODEL_ROOT = Path.cwd() / 'models_runtime'
PAPER_EXPORT_ROOT = OUTPUT_ROOT / 'paper_slide_model_exports'
PAPER_EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

print('Notebook folder:', Path.cwd())
print('Output root:', OUTPUT_ROOT)
print('Model root:', MODEL_ROOT)
print('Paper/slide export root:', PAPER_EXPORT_ROOT)

## 1. Find completed model runs

This scans every `run_manifest.json` under `outputs_runtime/`. A run is usually the main three-dataset run, dataset-3-as-training run, or all-saturation target run.

In [ ]:
def safe_read_json(path):
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except Exception as exc:
        return {'status': 'unreadable', 'error': str(exc), 'path': str(path)}

manifest_paths = sorted(OUTPUT_ROOT.rglob('run_manifest.json')) if OUTPUT_ROOT.exists() else []
manifest_rows = []
for path in manifest_paths:
    payload = safe_read_json(path)
    outputs = payload.get('outputs', {}) if isinstance(payload.get('outputs', {}), dict) else {}
    manifest_rows.append({
        'run_folder': str(path.parent),
        'manifest_path': str(path),
        'status': payload.get('status', ''),
        'blocked_reason': payload.get('blocked_reason', ''),
        'target_column': (payload.get('target', {}) or {}).get('column', payload.get('target_column', '')) if isinstance(payload.get('target', {}), dict) else payload.get('target', ''),
        'task': (payload.get('target', {}) or {}).get('task', payload.get('task', '')) if isinstance(payload.get('target', {}), dict) else '',
        'model_kind': payload.get('model_kind', ''),
        'feature_count': payload.get('feature_count', ''),
        'train_file': payload.get('train_file', ''),
        'test_files': ', '.join(map(str, payload.get('test_files', []))) if isinstance(payload.get('test_files', []), list) else payload.get('test_files', ''),
        'model_path': outputs.get('model', payload.get('model_path', '')),
    })

run_inventory = pd.DataFrame(manifest_rows)
run_inventory.to_csv(PAPER_EXPORT_ROOT / 'model_run_inventory.csv', index=False)
display(run_inventory)

## 2. Collect train/test metrics

This creates one combined metrics table for manuscript methods/results and slide summaries.

In [ ]:
metric_rows = []
for run_dir in sorted(OUTPUT_ROOT.iterdir()) if OUTPUT_ROOT.exists() else []:
    if not run_dir.is_dir():
        continue
    for metric_name in ['train_metrics.csv', 'test_metrics.csv', 'run_summary.csv']:
        path = run_dir / metric_name
        if path.exists():
            try:
                df = pd.read_csv(path)
                df.insert(0, 'run_folder', str(run_dir))
                df.insert(1, 'metric_file', metric_name)
                metric_rows.append(df)
            except Exception as exc:
                metric_rows.append(pd.DataFrame([{'run_folder': str(run_dir), 'metric_file': metric_name, 'read_error': str(exc)}]))

combined_metrics = pd.concat(metric_rows, ignore_index=True, sort=False) if metric_rows else pd.DataFrame()
combined_metrics.to_csv(PAPER_EXPORT_ROOT / 'combined_model_metrics.csv', index=False)
display(combined_metrics.head(100))

## 3. Inventory prediction files without exposing rows

This records prediction file locations, row counts, columns, target names, and dataset names. It avoids displaying full prediction rows by default.

In [ ]:
prediction_rows = []
for path in sorted(OUTPUT_ROOT.rglob('predictions_*.csv')) if OUTPUT_ROOT.exists() else []:
    try:
        df = pd.read_csv(path, nrows=5)
        full_rows = sum(1 for _ in open(path, 'r', encoding='utf-8', errors='ignore')) - 1
        prediction_rows.append({
            'prediction_file': str(path),
            'run_folder': str(path.parent),
            'rows': max(full_rows, 0),
            'columns': ', '.join(map(str, df.columns)),
            'target_column_sample': df['target_column'].dropna().iloc[0] if 'target_column' in df and df['target_column'].dropna().size else '',
            'dataset_file_sample': df['dataset_file'].dropna().iloc[0] if 'dataset_file' in df and df['dataset_file'].dropna().size else '',
            'has_y_true': 'y_true' in df.columns,
            'has_depth': 'depth_m' in df.columns,
            'has_well_alias': 'well_alias' in df.columns,
        })
    except Exception as exc:
        prediction_rows.append({'prediction_file': str(path), 'read_error': str(exc)})

prediction_inventory = pd.DataFrame(prediction_rows)
prediction_inventory.to_csv(PAPER_EXPORT_ROOT / 'prediction_file_inventory.csv', index=False)
display(prediction_inventory.head(100))

## 4. Feature columns and feature policy summaries

This combines `feature_columns.csv`, `feature_policy_audit.csv`, and all-saturation feature audit files so the paper can explain what entered the model and what was excluded as target leakage/context.

In [ ]:
feature_frames = []
audit_frames = []
for path in sorted(OUTPUT_ROOT.rglob('feature_columns.csv')) if OUTPUT_ROOT.exists() else []:
    try:
        df = pd.read_csv(path)
        df.insert(0, 'run_folder', str(path.parent))
        feature_frames.append(df)
    except Exception as exc:
        feature_frames.append(pd.DataFrame([{'run_folder': str(path.parent), 'read_error': str(exc)}]))
for name in ['feature_policy_audit.csv', 'excluded_feature_columns_by_target.csv']:
    for path in sorted(OUTPUT_ROOT.rglob(name)) if OUTPUT_ROOT.exists() else []:
        try:
            df = pd.read_csv(path)
            df.insert(0, 'run_folder', str(path.parent))
            df.insert(1, 'audit_file', name)
            audit_frames.append(df)
        except Exception as exc:
            audit_frames.append(pd.DataFrame([{'run_folder': str(path.parent), 'audit_file': name, 'read_error': str(exc)}]))

features_combined = pd.concat(feature_frames, ignore_index=True, sort=False) if feature_frames else pd.DataFrame()
audits_combined = pd.concat(audit_frames, ignore_index=True, sort=False) if audit_frames else pd.DataFrame()
features_combined.to_csv(PAPER_EXPORT_ROOT / 'combined_feature_columns.csv', index=False)
audits_combined.to_csv(PAPER_EXPORT_ROOT / 'combined_feature_policy_audits.csv', index=False)

display(features_combined.head(100))
display(audits_combined.head(100))

## 5. Extract feature importance from saved trained models

For random forest models, this exports feature importances. For MLP models, this records that direct impurity importance is not available.

In [ ]:
importance_rows = []
model_paths = []
if MODEL_ROOT.exists():
    model_paths.extend(sorted(MODEL_ROOT.rglob('model.joblib')))

for model_path in model_paths:
    try:
        payload = load(model_path)
        model = payload.get('model') if isinstance(payload, dict) else payload
        feature_columns = payload.get('feature_columns', payload.get('features', [])) if isinstance(payload, dict) else []
        target = payload.get('target', {}) if isinstance(payload, dict) else {}
        final_model = model.named_steps.get('model') if hasattr(model, 'named_steps') and 'model' in model.named_steps else model
        if hasattr(final_model, 'feature_importances_') and feature_columns:
            for feature, importance in zip(feature_columns, final_model.feature_importances_):
                importance_rows.append({
                    'model_path': str(model_path),
                    'run_folder': str(model_path.parent),
                    'target_column': target.get('column', target) if isinstance(target, dict) else target,
                    'task': target.get('task', '') if isinstance(target, dict) else '',
                    'feature_column': feature,
                    'importance': float(importance),
                    'importance_type': 'random_forest_impurity_importance',
                })
        else:
            importance_rows.append({
                'model_path': str(model_path),
                'run_folder': str(model_path.parent),
                'target_column': target.get('column', target) if isinstance(target, dict) else target,
                'task': target.get('task', '') if isinstance(target, dict) else '',
                'feature_column': '',
                'importance': np.nan,
                'importance_type': 'not_available_for_this_model_type',
            })
    except Exception as exc:
        importance_rows.append({'model_path': str(model_path), 'read_error': str(exc)})

feature_importance = pd.DataFrame(importance_rows)
feature_importance.to_csv(PAPER_EXPORT_ROOT / 'model_feature_importance.csv', index=False)
display(feature_importance.head(100))

## 6. Create paper/slide figures from outputs

These are lightweight PNGs for Word/PowerPoint. They do not expose raw approved input rows unless you choose to add those plots manually.

In [ ]:
figure_manifest = []
if plt is None:
    print('matplotlib is not available; skipping figure export.')
else:
    if not combined_metrics.empty:
        metric_cols = [c for c in ['mae','rmse','r2','accuracy','balanced_accuracy','f1_macro'] if c in combined_metrics.columns]
        for metric in metric_cols:
            plot_df = combined_metrics.dropna(subset=[metric]).copy()
            if plot_df.empty:
                continue
            plot_df['label'] = plot_df.get('dataset', plot_df.get('target_column', pd.Series(range(len(plot_df))))).astype(str) + ' / ' + plot_df.get('split', '').astype(str)
            ax = plot_df.plot(kind='bar', x='label', y=metric, legend=False, figsize=(10, 5))
            ax.set_title(f'Model {metric} by dataset/split')
            ax.set_xlabel('Dataset / split')
            ax.set_ylabel(metric)
            plt.xticks(rotation=45, ha='right')
            plt.tight_layout()
            out_path = PAPER_EXPORT_ROOT / f'figure_model_{metric}.png'
            plt.savefig(out_path, dpi=200)
            plt.close()
            figure_manifest.append({'figure': str(out_path), 'source': 'combined_model_metrics.csv', 'metric': metric})
    if not feature_importance.empty and 'importance' in feature_importance.columns:
        imp = feature_importance.dropna(subset=['importance']).sort_values('importance', ascending=False).head(20)
        if not imp.empty:
            ax = imp.plot(kind='bar', x='feature_column', y='importance', legend=False, figsize=(10, 5))
            ax.set_title('Top model feature importances')
            ax.set_xlabel('Feature')
            ax.set_ylabel('Importance')
            plt.xticks(rotation=45, ha='right')
            plt.tight_layout()
            out_path = PAPER_EXPORT_ROOT / 'figure_top_feature_importance.png'
            plt.savefig(out_path, dpi=200)
            plt.close()
            figure_manifest.append({'figure': str(out_path), 'source': 'model_feature_importance.csv', 'metric': 'importance'})

figure_manifest = pd.DataFrame(figure_manifest)
figure_manifest.to_csv(PAPER_EXPORT_ROOT / 'figure_manifest.csv', index=False)
display(figure_manifest)

## 7. Final deliverable manifest

This lists the output files to use in the research paper and slide deck.

In [ ]:
deliverables = []
for path in sorted(PAPER_EXPORT_ROOT.glob('*')):
    if path.is_file():
        deliverables.append({'file': str(path), 'type': path.suffix.lower().lstrip('.'), 'bytes': path.stat().st_size})

deliverable_manifest = pd.DataFrame(deliverables)
deliverable_manifest.to_csv(PAPER_EXPORT_ROOT / 'paper_slide_deliverable_manifest.csv', index=False)
print('Paper/slide outputs written to:', PAPER_EXPORT_ROOT)
display(deliverable_manifest)